In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt

def label_data(df):
    df.columns = df.columns.str.strip().str.lower()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    df["return"] = df["close"].pct_change().shift(-1)
    df["label"] = df["return"].apply(lambda x: 1 if x > 0.01 else (-1 if x < -0.01 else 0))
    df.dropna(inplace=True)

    return df

ta_data_path = "2.TA_data"
csv_files = glob.glob(os.path.join(ta_data_path, "*_TA.csv"))

labeled_data_path = "3.Labeled_data"
os.makedirs(labeled_data_path, exist_ok=True)  

for file in csv_files:
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip().str.lower()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

    df_labeled = label_data(df)
    label_counts = df_labeled["label"].value_counts(normalize=True) * 100  
    print(f"\n📊 {file} 라벨 비율 (%):")
    print(label_counts)

    plt.figure(figsize=(5, 3))
    label_counts.plot(kind="bar", color=["blue", "green", "red"])
    plt.xticks(ticks=[0, 1, 2], labels=["Hold (0)", "Buy (1)", "Sell (-1)"], rotation=0)
    plt.xlabel("Label")
    plt.ylabel("Percentage (%)")
    plt.title(f"Label Distribution - {os.path.basename(file)}")
    plt.show()

    new_filename = os.path.join(labeled_data_path, os.path.basename(file).replace("_TA.csv", "_Labeled.csv"))

    df_labeled.to_csv(new_filename, index=False)
    print(f"✅ {new_filename} 저장 완료!")


ValueError: Missing column provided to 'parse_dates': 'date'